# Create PoseNet CSV files from videos

This notebook generates PoseNet 2D keypoint CSV files from the videos in `data/raw/all_videos`.

Output:
- one CSV per video
- saved to `data/raw/posenet_generated`

The generated CSV files are then used as input to A10:
- PoseNet 2D -> Kinect 2D

### Imports and paths

In [37]:
from pathlib import Path
import os
import sys
import json
import time
import random
import warnings

import numpy as np
import pandas as pd
import cv2
import torch
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

VIDEO_DIR = Path("../../data/raw/all_videos")
OUTPUT_DIR = Path("../../data/raw/posenet_generated")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("DEVICE:", DEVICE)
print("VIDEO_DIR exists:", VIDEO_DIR.exists(), VIDEO_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

DEVICE: cuda
VIDEO_DIR exists: True ..\..\data\raw\all_videos
OUTPUT_DIR: ..\..\data\raw\posenet_generated


### Install PoseNet notebook dependencies

In [38]:
import sys
from pathlib import Path

POSENET_REPO = Path("../../../posenet-pytorch").resolve()
if str(POSENET_REPO) not in sys.path:
    sys.path.insert(0, str(POSENET_REPO))

print(POSENET_REPO)

C:\Users\Kim\software engineering\year 4\LP3\4DT907\machine_learning_project_4dt907\ml\posenet-pytorch


In [39]:
import sys
import torch

print(sys.executable)
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.version.cuda)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no gpu")

c:\Users\Kim\software engineering\year 4\LP3\4DT907\machine_learning_project_4dt907\.venv\Scripts\python.exe
2.7.1+cu118
True
11.8
NVIDIA GeForce RTX 2070


In [40]:
import posenet
from posenet.decode_multi import decode_multiple_poses

print("PoseNet import worked")
print("PoseNet module:", posenet.__file__)

PoseNet import worked
PoseNet module: C:\Users\Kim\software engineering\year 4\LP3\4DT907\machine_learning_project_4dt907\ml\posenet-pytorch\posenet\__init__.py


### Config

In [41]:
MODEL_ID = 101
SCALE_FACTOR = 1.0
MAX_POSE_DETECTIONS = 5
MIN_POSE_SCORE = 0.05
MIN_KEYPOINT_SCORE = 0.0

PROJECT_JOINTS = [
    "head",
    "left_shoulder",
    "left_elbow",
    "right_shoulder",
    "right_elbow",
    "left_hand",
    "right_hand",
    "left_hip",
    "right_hip",
    "left_knee",
    "right_knee",
    "left_foot",
    "right_foot",
]

POSENET_KEYPOINT_NAMES = [
    "nose",
    "left_eye",
    "right_eye",
    "left_ear",
    "right_ear",
    "left_shoulder",
    "right_shoulder",
    "left_elbow",
    "right_elbow",
    "left_wrist",
    "right_wrist",
    "left_hip",
    "right_hip",
    "left_knee",
    "right_knee",
    "left_ankle",
    "right_ankle",
]

PROJECT_TO_POSENET = {
    "head": "nose",
    "left_shoulder": "left_shoulder",
    "left_elbow": "left_elbow",
    "right_shoulder": "right_shoulder",
    "right_elbow": "right_elbow",
    "left_hand": "left_wrist",
    "right_hand": "right_wrist",
    "left_hip": "left_hip",
    "right_hip": "right_hip",
    "left_knee": "left_knee",
    "right_knee": "right_knee",
    "left_foot": "left_ankle",
    "right_foot": "right_ankle",
}

POSENET_NAME_TO_INDEX = {name: idx for idx, name in enumerate(POSENET_KEYPOINT_NAMES)}

model = posenet.load_model(MODEL_ID)
model = model.to(DEVICE)
model.eval()

output_stride = model.output_stride

print("Loaded PoseNet model:", MODEL_ID)
print("Output stride:", output_stride)
print("Device:", DEVICE)

Loaded PoseNet model: 101
Output stride: 16
Device: cuda


### Load model on GPU

In [42]:
model = posenet.load_model(MODEL_ID)
model = model.to(DEVICE)
model.eval()

output_stride = model.output_stride
print("Loaded PoseNet model:", MODEL_ID)
print("Output stride:", output_stride)
print("Device:", DEVICE)

Loaded PoseNet model: 101
Output stride: 16
Device: cuda


### Helper functions

In [43]:
def extract_base_id(video_path: Path) -> str:
    return video_path.stem


def choose_best_pose(pose_scores, keypoint_scores):
    if pose_scores is None or len(pose_scores) == 0:
        return None

    best_idx = None
    best_tuple = None

    for i in range(len(pose_scores)):
        pose_score = float(pose_scores[i])
        mean_kp = float(np.mean(keypoint_scores[i]))
        score_tuple = (pose_score, mean_kp)

        if best_tuple is None or score_tuple > best_tuple:
            best_tuple = score_tuple
            best_idx = i

    return best_idx


def keypoints_to_project_row(frame_no, pose_score, keypoint_scores, keypoint_coords):
    row = {
        "frameno": int(frame_no),
        "pose_score": float(pose_score),
    }

    for project_joint in PROJECT_JOINTS:
        posenet_name = PROJECT_TO_POSENET[project_joint]
        kp_idx = POSENET_NAME_TO_INDEX[posenet_name]

        y_coord = float(keypoint_coords[kp_idx, 0])
        x_coord = float(keypoint_coords[kp_idx, 1])
        kp_score = float(keypoint_scores[kp_idx])

        # always save coordinates
        row[f"{project_joint}_x"] = x_coord
        row[f"{project_joint}_y"] = y_coord
        row[f"{project_joint}_score"] = kp_score

    return row


def make_empty_row(frame_no):
    row = {
        "frameno": int(frame_no),
        "pose_score": np.nan,
    }
    for joint in PROJECT_JOINTS:
        row[f"{joint}_x"] = np.nan
        row[f"{joint}_y"] = np.nan
        row[f"{joint}_score"] = np.nan
    return row


def process_video_to_posenet_csv(video_path: Path, output_dir: Path, overwrite: bool = False):
    base_id = extract_base_id(video_path)
    output_csv = output_dir / f"{base_id}_posenet.csv"

    if output_csv.exists() and not overwrite:
        return {
            "video": video_path.name,
            "status": "skipped_exists",
            "frames_total": None,
            "frames_with_pose": None,
            "output_csv": str(output_csv),
        }

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return {
            "video": video_path.name,
            "status": "failed_open",
            "frames_total": None,
            "frames_with_pose": None,
            "output_csv": str(output_csv),
        }

    rows = []
    frame_no = 0
    frames_with_pose = 0

    try:
        while True:
            ok, frame = cap.read()
            if not ok:
                break

            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            input_image, display_image, output_scale = posenet.process_input(
                rgb_frame,
                scale_factor=SCALE_FACTOR,
                output_stride=output_stride,
            )

            input_image = torch.tensor(input_image, dtype=torch.float32, device=DEVICE)

            with torch.no_grad():
                heatmaps_result, offsets_result, displacement_fwd_result, displacement_bwd_result = model(input_image)

                pose_scores, keypoint_scores, keypoint_coords = posenet.decode_multiple_poses(
                    heatmaps_result.squeeze(0),
                    offsets_result.squeeze(0),
                    displacement_fwd_result.squeeze(0),
                    displacement_bwd_result.squeeze(0),
                    output_stride=output_stride,
                    max_pose_detections=MAX_POSE_DETECTIONS,
                    min_pose_score=MIN_POSE_SCORE,
                )

            keypoint_coords *= output_scale
            best_idx = choose_best_pose(pose_scores, keypoint_scores)

            if best_idx is None or float(pose_scores[best_idx]) < MIN_POSE_SCORE:
                rows.append(make_empty_row(frame_no))
            else:
                row = keypoints_to_project_row(
                    frame_no=frame_no,
                    pose_score=pose_scores[best_idx],
                    keypoint_scores=keypoint_scores[best_idx],
                    keypoint_coords=keypoint_coords[best_idx],
                )
                rows.append(row)
                frames_with_pose += 1

            frame_no += 1

    finally:
        cap.release()

    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False)

    return {
        "video": video_path.name,
        "status": "ok",
        "frames_total": int(frame_no),
        "frames_with_pose": int(frames_with_pose),
        "output_csv": str(output_csv),
    }

### Single-video processor

In [44]:
def process_video_to_posenet_csv(video_path: Path, output_dir: Path, overwrite: bool = False):
    base_id = extract_base_id(video_path)
    output_csv = output_dir / f"{base_id}_posenet.csv"

    if output_csv.exists() and not overwrite:
        return {
            "video": video_path.name,
            "status": "skipped_exists",
            "frames_total": None,
            "frames_with_pose": None,
            "output_csv": str(output_csv),
        }

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return {
            "video": video_path.name,
            "status": "failed_open",
            "frames_total": None,
            "frames_with_pose": None,
            "output_csv": str(output_csv),
        }

    rows = []
    frame_no = 0
    frames_with_pose = 0

    try:
        while True:
            try:
                read_result = posenet.read_cap(
                    cap,
                    scale_factor=SCALE_FACTOR,
                    output_stride=output_stride,
                )
            except Exception as e:
                # Treat webcam failure as normal end-of-video if we already processed frames
                if "webcam failure" in str(e).lower() and frame_no > 0:
                    break
                raise

            if read_result is None:
                break

            input_image, display_image, output_scale = read_result

            if input_image is None:
                break

            input_image = torch.tensor(input_image, dtype=torch.float32, device=DEVICE)

            with torch.no_grad():
                heatmaps_result, offsets_result, displacement_fwd_result, displacement_bwd_result = model(input_image)

                pose_scores, keypoint_scores, keypoint_coords, _ = decode_multiple_poses(
                    heatmaps_result.squeeze(0),
                    offsets_result.squeeze(0),
                    displacement_fwd_result.squeeze(0),
                    displacement_bwd_result.squeeze(0),
                    output_stride=output_stride,
                    max_pose_detections=MAX_POSE_DETECTIONS,
                    min_pose_score=MIN_POSE_SCORE,
                )

            keypoint_coords *= output_scale
            best_idx = choose_best_pose(pose_scores, keypoint_scores)

            if best_idx is None or float(pose_scores[best_idx]) < MIN_POSE_SCORE:
                rows.append(make_empty_row(frame_no))
            else:
                row = keypoints_to_project_row(
                    frame_no=frame_no,
                    pose_score=pose_scores[best_idx],
                    keypoint_scores=keypoint_scores[best_idx],
                    keypoint_coords=keypoint_coords[best_idx],
                )
                rows.append(row)
                frames_with_pose += 1

            frame_no += 1

    except Exception as e:
        cap.release()
        print(f"Error while processing {video_path.name}: {e}")
        return {
            "video": video_path.name,
            "status": f"failed_runtime: {e}",
            "frames_total": int(frame_no),
            "frames_with_pose": int(frames_with_pose),
            "output_csv": str(output_csv),
        }

    cap.release()

    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False)

    return {
        "video": video_path.name,
        "status": "ok",
        "frames_total": int(frame_no),
        "frames_with_pose": int(frames_with_pose),
        "output_csv": str(output_csv),
    }

### Quick single-video test

In [45]:
video_files = sorted([p for p in VIDEO_DIR.glob("*.avi") if p.is_file()])
print("Video count:", len(video_files))
print("First 10 videos:", [p.name for p in video_files[:10]])

test_video = video_files[0]
print("Testing:", test_video)

test_result = process_video_to_posenet_csv(test_video, OUTPUT_DIR, overwrite=True)
print(test_result)

Video count: 180
First 10 videos: ['A1.avi', 'A10.avi', 'A100.avi', 'A101.avi', 'A102.avi', 'A103.avi', 'A104.avi', 'A105.avi', 'A106.avi', 'A108.avi']
Testing: ..\..\data\raw\all_videos\A1.avi
{'video': 'A1.avi', 'status': 'ok', 'frames_total': 229, 'frames_with_pose': 195, 'output_csv': '..\\..\\data\\raw\\posenet_generated\\A1_posenet.csv'}


### Inspect one generated CSV

In [46]:
test_csv = OUTPUT_DIR / f"{test_video.stem}_posenet.csv"
test_df = pd.read_csv(test_csv)

print(test_csv)
print(test_df.shape)
display(test_df.head())

..\..\data\raw\posenet_generated\A1_posenet.csv
(229, 41)


,frameno,pose_score,head_x,head_y,head_score,left_shoulder_x,left_shoulder_y,left_shoulder_score,left_elbow_x,left_elbow_y,...,left_knee_score,right_knee_x,right_knee_y,right_knee_score,left_foot_x,left_foot_y,left_foot_score,right_foot_x,right_foot_y,right_foot_score
0,0,0.581697,996.023887,265.084843,0.996027,1055.432203,376.018032,0.123611,1079.204110,504.769206,...,0.556332,916.997710,798.007567,0.021123,1045.164827,994.553931,0.512636,913.995613,981.430887,0.381219
1,1,0.522652,995.816585,265.831209,0.997706,1057.117063,375.935866,0.178634,1081.560528,500.670052,...,0.314646,919.038782,797.981085,0.026792,1046.589077,994.955285,0.470414,915.519810,988.779386,0.348525
2,2,0.561555,995.958964,265.213109,0.997457,1066.624843,375.525101,0.439956,1078.293309,499.282978,...,0.569471,918.777596,798.874512,0.029565,1046.894612,995.491196,0.515324,916.258987,981.139165,0.225078
3,3,0.560089,995.595493,266.096842,0.998052,1057.401481,375.056845,0.271424,1080.000343,499.237994,...,0.507430,914.357436,811.492775,0.019608,1047.554681,1002.468348,0.610233,913.668208,993.443761,0.423945
4,4,0.527051,995.604127,265.939714,0.997914,1057.855995,374.944033,0.310070,1079.428696,498.299584,...,0.434066,918.585506,798.491042,0.034241,1047.294639,995.537836,0.520567,915.086778,990.044714,0.320184


### Process all videos

In [47]:
video_files = sorted([p for p in VIDEO_DIR.glob("*.avi") if p.is_file()])

results = []
start_time = time.time()

for video_path in tqdm(video_files, desc="Processing videos"):
    result = process_video_to_posenet_csv(video_path, OUTPUT_DIR, overwrite=False)
    results.append(result)

elapsed = time.time() - start_time

results_df = pd.DataFrame(results)
print(f"Finished in {elapsed/60:.2f} minutes")
display(results_df.head())

Processing videos: 100%|██████████| 180/180 [37:43<00:00, 12.57s/it]

Finished in 37.72 minutes


,video,status,frames_total,frames_with_pose,output_csv
0,A1.avi,skipped_exists,NaN,NaN,..\..\data\raw\posenet_generated\A1_posenet.csv
1,A10.avi,ok,257.0,237.0,..\..\data\raw\posenet_generated\A10_posenet.csv
2,A100.avi,ok,223.0,223.0,..\..\data\raw\posenet_generated\A100_posenet.csv
3,A101.avi,ok,272.0,272.0,..\..\data\raw\posenet_generated\A101_posenet.csv
4,A102.avi,ok,212.0,212.0,..\..\data\raw\posenet_generated\A102_posenet.csv


### Summary of generation run

In [48]:
print("Status counts:")
display(results_df["status"].value_counts(dropna=False))

ok_df = results_df[results_df["status"] == "ok"].copy()
if not ok_df.empty:
    ok_df["pose_detection_rate"] = ok_df["frames_with_pose"] / ok_df["frames_total"]
    display(ok_df[["video", "frames_total", "frames_with_pose", "pose_detection_rate"]].head())
    print("Mean pose detection rate:", ok_df["pose_detection_rate"].mean())

Status counts:


status
ok                179
skipped_exists      1
Name: count, dtype: int64

,video,frames_total,frames_with_pose,pose_detection_rate
1,A10.avi,257.0,237.0,0.922179
2,A100.avi,223.0,223.0,1.000000
3,A101.avi,272.0,272.0,1.000000
4,A102.avi,212.0,212.0,1.000000
5,A103.avi,280.0,280.0,1.000000


Mean pose detection rate: 0.9400326661463054


### Save run summary

In [49]:
summary_csv = OUTPUT_DIR / "posenet_generation_summary.csv"
summary_json = OUTPUT_DIR / "posenet_generation_summary.json"

results_df.to_csv(summary_csv, index=False)

with open(summary_json, "w") as f:
    json.dump(results, f, indent=2)

print("Saved summary CSV:", summary_csv)
print("Saved summary JSON:", summary_json)

Saved summary CSV: ..\..\data\raw\posenet_generated\posenet_generation_summary.csv
Saved summary JSON: ..\..\data\raw\posenet_generated\posenet_generation_summary.json


### Verify filenames match Kinect IDs

In [50]:
KINECT_DIR = Path("../../data/raw/kinect_good_preprocessed")

kinect_ids = {p.stem.replace("_kinect", "") for p in KINECT_DIR.glob("*.csv")}
posenet_ids = {p.stem.replace("_posenet", "") for p in OUTPUT_DIR.glob("*.csv") if p.name != "posenet_generation_summary.csv"}

common_ids = sorted(kinect_ids & posenet_ids)
missing_in_posenet = sorted(kinect_ids - posenet_ids)
missing_in_kinect = sorted(posenet_ids - kinect_ids)

print("Matched IDs:", len(common_ids))
print("Missing in PoseNet:", missing_in_posenet[:20])
print("Missing in Kinect:", missing_in_kinect[:20])

Matched IDs: 179
Missing in PoseNet: []
Missing in Kinect: ['A60']


### Final ready-check for A10

In [51]:
sample_output_files = sorted([p.name for p in OUTPUT_DIR.glob("*_posenet.csv")])[:10]
print("Sample generated PoseNet files:")
print(sample_output_files)

print("\nA10 should use:")
print("PoseNet dir:", OUTPUT_DIR)
print("Kinect dir:", KINECT_DIR)

Sample generated PoseNet files:
['A100_posenet.csv', 'A101_posenet.csv', 'A102_posenet.csv', 'A103_posenet.csv', 'A104_posenet.csv', 'A105_posenet.csv', 'A106_posenet.csv', 'A108_posenet.csv', 'A109_posenet.csv', 'A10_posenet.csv']

A10 should use:
PoseNet dir: ..\..\data\raw\posenet_generated
Kinect dir: ..\..\data\raw\kinect_good_preprocessed
